# 02 Pipeline

Build a governed, quality-checked workflow with visible **Environment → Extract → Transform → Load** steps. Every source and target begins with an existing FabricOps Catalogue selection and retains its canonical `table_id`.

The template runs with one source and one target. Its indexed dictionaries make complete SOURCE or TARGET blocks cloneable without numbered Python variable names. FabricOps recommends one governed target per pipeline.

## Tested with FabricOps

The previous baseline of this template was run in Microsoft Fabric with FabricOps v0.2.0 by Voyce on 6 Aug 2026. This redesigned workflow has local structural and public-API compatibility validation only; run it in your configured Fabric workspace before treating it as runtime-validated.

# 0. Environment

Run the shared configuration, then import only the public functions used by this template.

In [ ]:
%run 00_env_config

In [ ]:
from pyspark.sql import functions as F

from fabricops_kit import (
    check_dq,
    check_freshness,
    check_schema,
    profile_and_register_table,
    profile_dataframe,
    read_lakehouse_csv,
    read_lakehouse_excel,
    read_lakehouse_parquet,
    read_lakehouse_table,
    read_pipeline_prep,
    read_warehouse_query,
    read_warehouse_table,
    write_lakehouse_table,
    write_pipeline_prep,
    write_warehouse_table,
    widget_select_data_contract,
    widget_view_catalogue,
)

# E. Extract

Initialize the shared indexed workflow state once. Integer keys keep each cloned block's Catalogue identity, preparation state, DataFrame, Guardrail results, profile, and Data Contract independent.

In [ ]:
SOURCES = {}
SOURCE_PREPS = {}
SOURCE_DFS = {}
SOURCE_PROFILES = {}
SOURCE_RESULTS = {}

TARGETS = {}
TARGET_DFS = {}
TARGET_PREPS = {}
TARGET_RESULTS = {}
TARGET_CONTRACTS = {}

## SOURCE 1 — Select and configure

Select one registered source from the active FabricOps Catalogue. The returned `table_id` is authoritative; never reconstruct it from physical coordinates. Then keep this source's engineer-owned processing settings visible.

In [ ]:
SOURCE = 1

source_catalogue = widget_view_catalogue(
    mode="explore",
    spark_session=spark,
)
source_selection = source_catalogue["get_selection"]()

SOURCES[SOURCE] = {
    "table_id": source_selection["table_id"],
    "store_type": source_selection["store_type"],
    "target": source_selection["layer"],
    "schema": source_selection["schema_name"],
    "table_name": source_selection["table_name"],
}
if not SOURCES[SOURCE]["table_id"]:
    raise ValueError("Select a registered source table from the Catalogue first.")

source = SOURCES[SOURCE]
source["read_strategy"] = "incremental_watermark"
source["watermark_column"] = "modified_datetime"
source["partition_column"] = None

# Other valid source settings:
# source["read_strategy"] = "full_dataset"
# source["watermark_column"] = None
# source["read_strategy"] = "incremental_partition"
# source["partition_column"] = "snapshot_date"

## TARGET 1 — Select, configure, and choose its Data Contract

`read_pipeline_prep()` resolves target processing as well as source scope, so select the registered target before preparation. Development uses current mutable authoring or one exact frozen Data Contract version; Production automatically uses exactly one active Data Contract.

In [ ]:
TARGET = 1

target_catalogue = widget_view_catalogue(
    mode="explore",
    spark_session=spark,
)
target_selection = target_catalogue["get_selection"]()

TARGETS[TARGET] = {
    "table_id": target_selection["table_id"],
    "store_type": target_selection["store_type"],
    "target": target_selection["layer"],
    "schema": target_selection["schema_name"],
    "table_name": target_selection["table_name"],
    "load_strategy": "scd1",
    "load_parameters": {"key_columns": ["student_id"]},
}
if not TARGETS[TARGET]["table_id"]:
    raise ValueError("Select a registered target table from the Catalogue first.")

target = TARGETS[TARGET]
TARGET_CONTRACTS[TARGET] = widget_select_data_contract(table_id=target["table_id"])

source = SOURCES[SOURCE]
SOURCE_PREPS[SOURCE] = read_pipeline_prep(
    source_table_id=source["table_id"],
    source_table_name=source["table_name"],
    target_table_name=target["table_name"],
    source_read_strategy=source["read_strategy"],
    source_watermark_column=source["watermark_column"],
    source_partition_column=source["partition_column"],
    source_target=source["target"],
    source_schema=source["schema"],
    target=target["target"],
    schema=target["schema"],
    target_table_id=target["table_id"],
    load_strategy=target["load_strategy"],
    load_strategy_parameters=target["load_parameters"],
)

prep = SOURCE_PREPS[SOURCE]
schema_result = check_schema(
    source["table_name"], target=source["target"], schema=source["schema"],
)
SOURCE_RESULTS[SOURCE] = {"pre_read": [schema_result]}
if prep["observation"] is not None:
    SOURCE_RESULTS[SOURCE]["pre_read"].append(check_freshness(prep["observation"]))
if prep["changes"] is not None:
    SOURCE_RESULTS[SOURCE]["pre_read"].append(prep["changes"])
if not all(result["can_continue"] for result in SOURCE_RESULTS[SOURCE]["pre_read"]):
    raise RuntimeError("A source Guardrail blocked this run.")

SOURCE_RESULTS[SOURCE]["should_run"] = prep["read_mode"] != "skip"
print(f'Runtime read mode: {prep["read_mode"]}')

## SOURCE 1 — Read

The Catalogue `store_type` chooses the explicit physical table reader. The prepared scope remains visible and a skip decision bypasses downstream work.

In [ ]:
source = SOURCES[SOURCE]
prep = SOURCE_PREPS[SOURCE]

if not SOURCE_RESULTS[SOURCE]["should_run"]:
    SOURCE_DFS[SOURCE] = None
    print("No source changes detected. Source read and downstream publication are skipped.")
elif source["store_type"] == "warehouse":
    SOURCE_DFS[SOURCE] = read_warehouse_table(
        source["schema"], source["table_name"], target=source["target"],
        spark_session=spark, processing_scope=prep["scope"],
    )
elif source["store_type"] == "lakehouse":
    SOURCE_DFS[SOURCE] = read_lakehouse_table(
        source["table_name"], target=source["target"], schema=source["schema"],
        spark_session=spark, processing_scope=prep["scope"],
    )
else:
    raise ValueError(f'Unsupported Catalogue store_type: {source["store_type"]!r}')

## SOURCE 1 — DQ and Profile

DQ evaluates the rows selected for this run. A complete source read may replace the canonical registered source profile. An incremental slice is diagnostic only and must not replace the latest complete profile.

In [ ]:
source = SOURCES[SOURCE]
prep = SOURCE_PREPS[SOURCE]

if SOURCE_RESULTS[SOURCE]["should_run"]:
    SOURCE_RESULTS[SOURCE]["dq"] = check_dq(
        SOURCE_DFS[SOURCE], source["table_name"],
        target=source["target"], schema=source["schema"],
    )
    display(SOURCE_RESULTS[SOURCE]["dq"]["summary"])
    if not SOURCE_RESULTS[SOURCE]["dq"]["can_continue"]:
        raise RuntimeError("A source DQ Guardrail blocked this run.")

    if prep["read_mode"] == "full_dataset":
        SOURCE_PROFILES[SOURCE] = profile_and_register_table(
            SOURCE_DFS[SOURCE], profile_role="source", target=source["target"],
            schema=source["schema"], table_name=source["table_name"],
        )
    elif prep["read_mode"] == "incremental_subset":
        SOURCE_PROFILES[SOURCE] = profile_dataframe(SOURCE_DFS[SOURCE])

    display(SOURCE_PROFILES[SOURCE])

### Need another source?

Duplicate the complete SOURCE block and change `SOURCE = 2`. Each source keeps its own Catalogue identity, preparation state, DataFrame, Guardrail results, and profile state. Do not make Source 2 active unless the transformation needs it.

### Clone pattern

The selected Catalogue row determines whether the visible table read uses the Warehouse or Lakehouse reader. Registered table sources do not automatically imply Lakehouse Files behavior.

In [ ]:
# Clone the complete SOURCE block above, then change only its key and settings:
# SOURCE = 2

# T. Transform

**Business transformation is engineer-owned.** Keep joins, filters, derivations, aggregations, and reshaping visible rather than hiding them in a FabricOps orchestrator.

In [ ]:
if SOURCE_RESULTS[1]["should_run"]:
    transformed_df = SOURCE_DFS[1]
    display(transformed_df)

# Multi-source example after cloning and running SOURCE = 2:
# transformed_df = SOURCE_DFS[1].join(
#     SOURCE_DFS[2],
#     on="student_id",
#     how="left",
# )

# L. Load

## TARGET 1 — Prepare and Guard

Assign the engineer-owned transformed output explicitly. Each cloned target can publish a different DataFrame and selects its own Data Contract by canonical `table_id`.

In [ ]:
TARGET_DFS[1] = transformed_df

target = TARGETS[TARGET]
target_df = TARGET_DFS[TARGET]
prep = SOURCE_PREPS[SOURCE]

if SOURCE_RESULTS[SOURCE]["should_run"]:
    TARGET_RESULTS[TARGET] = {
        "schema": check_schema(
            target["table_name"], target=target["target"],
            schema=target["schema"], dataframe=target_df,
        ),
        "dq": check_dq(
            target_df, target["table_name"],
            target=target["target"], schema=target["schema"],
        ),
    }
    display(TARGET_RESULTS[TARGET]["dq"]["summary"])
    if not all(result["can_continue"] for result in TARGET_RESULTS[TARGET].values()):
        raise RuntimeError("A target Guardrail blocked publication.")

    TARGET_PREPS[TARGET] = write_pipeline_prep(
        target_df, prep, target=target["target"],
    )

## TARGET 1 — Publish

The physical writer stays explicit. The canonical one-target path preserves successful source completion only after its target write succeeds.

In [ ]:
target = TARGETS[TARGET]
write_prep = TARGET_PREPS.get(TARGET)

if SOURCE_RESULTS[SOURCE]["should_run"]:
    if target["store_type"] == "lakehouse":
        write_lakehouse_table(
            write_prep["df"], target["table_name"], target=target["target"],
            schema=target["schema"], mode=write_prep["mode"],
            options=write_prep["options"], load_strategy=write_prep["load_strategy"],
            load_strategy_parameters=write_prep["load_strategy_parameters"],
            processing_scope=write_prep["scope"],
            completion_context=write_prep["completion"],
        )
    elif target["store_type"] == "warehouse":
        write_warehouse_table(
            write_prep["df"], target["schema"], target["table_name"],
            target=target["target"], mode=write_prep["mode"],
            options=write_prep["options"], load_strategy=write_prep["load_strategy"],
            load_strategy_parameters=write_prep["load_strategy_parameters"],
            completion_context=write_prep["completion"],
        )
    else:
        raise ValueError(f'Unsupported Catalogue store_type: {target["store_type"]!r}')

### Completion limitation

Cloned target writes are independently committed and are not automatically rolled back together. The canonical path completes source checkpoints with its one target. If you clone TARGET blocks, review completion ownership explicitly; this template does not provide distributed transactions.

In [ ]:
# FabricOps recommends one governed target per pipeline.
# Multiple targets are supported when needed by cloning the complete TARGET block:
# TARGET = 2
# TARGET_DFS[2] = summary_df

### Need another target?

FabricOps recommends **one governed target per pipeline** because target writes are independently committed and are not automatically rolled back together. **Multiple targets are supported** when needed: duplicate the complete TARGET block, change `TARGET = 2`, select another registered target, assign `TARGET_DFS[2]`, and select that target's own Data Contract.

# Common patterns

| Source composition | Source strategy | Typical target strategy | Notes |
|---|---|---|---|
| Warehouse or Lakehouse table → target | `incremental_watermark` | `append`, `scd1`, or `scd2` | Use the Catalogue-selected table and governed scope. |
| Multiple registered sources → transform → target | workload-specific | workload-specific | Clone complete SOURCE blocks and join `SOURCE_DFS` explicitly. |
| One transform → multiple registered targets | workload-specific | workload-specific | Supported by cloned TARGET blocks; writes are independently committed. |